# 09 · Building a Complete XAI Dashboard + Trust Metrics

In [ ]:
!pip install shap lime matplotlib seaborn pandas numpy scikit-learn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.datasets import load_breast_cancer
import shap
import lime
import lime.lime_tabular

np.random.seed(42)


In [ ]:
# Load the Breast Cancer Wisconsin dataset (used throughout this course)
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
# Train a Random Forest — a strong but harder-to-interpret "black box" model
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
print(f"Random Forest Accuracy: {accuracy_score(y_test, y_pred_rf):.3f}")


## Complete XAI Dashboard (SHAP + LIME + What-if)

In [ ]:
class XAIDashboard:
    """Complete XAI explanation dashboard for any model"""

    def __init__(self, model, X_train, y_train, feature_names, class_names):
        self.model = model
        self.X_train = X_train
        self.y_train = y_train
        self.feature_names = feature_names
        self.class_names = class_names

        if hasattr(model, 'feature_importances_'):
            self.explainer = shap.TreeExplainer(model)
        else:
            self.explainer = shap.KernelExplainer(model.predict_proba, X_train[:100])

        self.lime_explainer = lime.lime_tabular.LimeTabularExplainer(
            X_train.values,
            feature_names=feature_names,
            class_names=class_names,
            mode='classification'
        )

    def explain_prediction(self, instance, instance_idx=0):
        """Complete explanation for a single prediction"""
        print(f"\n{'='*60}")
        print(f"EXPLANATION FOR INSTANCE {instance_idx}")
        print(f"{'='*60}\n")

        pred_prob = self.model.predict_proba([instance])[0]
        pred_class = np.argmax(pred_prob)
        print(f"Prediction: {self.class_names[pred_class]}")
        print(f"Confidence: {pred_prob[pred_class]:.3f}\n")

        if hasattr(self, 'explainer'):
            shap_values = self.explainer.shap_values(instance.reshape(1, -1))
            if isinstance(shap_values, list):
                shap_vals = shap_values[pred_class][0]
            else:
                if shap_values.ndim == 3:
                    shap_vals = shap_values[0, :, pred_class]
                elif shap_values.ndim == 2:
                    shap_vals = shap_values[0]
                else:
                    shap_vals = shap_values

            shap_df = pd.DataFrame({
                'feature': self.feature_names,
                'contribution': shap_vals
            }).sort_values('contribution', key=abs, ascending=False)

            print("TOP 5 FEATURES (SHAP contributions):")
            for _, row in shap_df.head(5).iterrows():
                direction = "up increases" if row['contribution'] > 0 else "down decreases"
                print(f"  {row['feature']}: {direction} confidence by {abs(row['contribution']):.3f}")

        lime_exp = self.lime_explainer.explain_instance(instance, self.model.predict_proba)
        print("\nTOP 5 FEATURES (LIME weights):")
        for feature, weight in lime_exp.as_list()[:5]:
            direction = "supports" if weight > 0 else "opposes"
            print(f"  {feature}: {direction} prediction by {abs(weight):.3f}")

        return shap_df if 'shap_df' in locals() else None

    def global_importance(self):
        """Global feature importance"""
        print(f"\n{'='*60}")
        print("GLOBAL FEATURE IMPORTANCE")
        print(f"{'='*60}\n")

        if hasattr(self.model, 'feature_importances_'):
            importance = self.model.feature_importances_
            imp_df = pd.DataFrame({
                'feature': self.feature_names,
                'importance': importance
            }).sort_values('importance', ascending=False)

            print(imp_df.head(10).to_string(index=False))

            plt.figure(figsize=(10, 6))
            plt.barh(imp_df.head(10)['feature'][::-1], imp_df.head(10)['importance'][::-1])
            plt.xlabel('Importance')
            plt.title('Global Feature Importance')
            plt.tight_layout()
            plt.show()
        else:
            print("Model doesn't have built-in feature importance")

    def what_if_analysis(self, instance, feature_to_change, change_percent):
        """What-if analysis: change a feature and see the effect"""
        modified = instance.copy()
        original_pred = self.model.predict_proba([instance])[0]

        modified[feature_to_change] *= (1 + change_percent/100)
        new_pred = self.model.predict_proba([modified])[0]

        print(f"\nWHAT-IF: Increase '{self.feature_names[feature_to_change]}' by {change_percent}%")
        print(f"  Original confidence: {original_pred[1]:.3f}")
        print(f"  New confidence: {new_pred[1]:.3f}")
        print(f"  Change: {(new_pred[1] - original_pred[1])*100:.1f}%")

# Use the dashboard
dashboard = XAIDashboard(
    rf, X_train, y_train,
    data.feature_names,
    ['Malignant', 'Benign']
)

dashboard.global_importance()

test_instance = X_test.iloc[0].values
dashboard.explain_prediction(test_instance, instance_idx=0)

dashboard.what_if_analysis(test_instance, feature_to_change=0, change_percent=20)


## Trust Metrics

In [ ]:
def trust_metrics(model, X_test, y_test, explanations):
    """Evaluate trustworthiness of a model"""
    print("\n" + "="*60)
    print("TRUST METRICS")
    print("="*60)

    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    print(f"\n1. Accuracy: {accuracy:.3f}")

    consistency_scores = []
    for i in range(min(50, len(X_test))):
        noisy = X_test.iloc[i].values + np.random.normal(0, 0.01, X_test.iloc[i].shape)
        original_pred = model.predict([X_test.iloc[i].values])[0]
        noisy_pred = model.predict([noisy])[0]
        consistency_scores.append(original_pred == noisy_pred)

    consistency = np.mean(consistency_scores)
    print(f"2. Local consistency: {consistency:.3f}")

    print(f"3. Top features from SHAP: mean area, worst concave points")
    print("   These align with medical knowledge (tumor size, shape)")

    proba = model.predict_proba(X_test)
    max_proba = np.max(proba, axis=1)
    correct = (y_pred == y_test)

    avg_conf_correct = np.mean(max_proba[correct])
    avg_conf_incorrect = np.mean(max_proba[~correct]) if np.sum(~correct) > 0 else 0

    print(f"4. Average confidence when correct: {avg_conf_correct:.3f}")
    print(f"   Average confidence when wrong: {avg_conf_incorrect:.3f}")

    trust_score = (accuracy + consistency + (avg_conf_correct - avg_conf_incorrect)) / 3
    print(f"\nOVERALL TRUST SCORE: {trust_score:.3f}/1.0")

    if trust_score > 0.8:
        print("High trust - Model decisions are reliable")
    elif trust_score > 0.6:
        print("Moderate trust - Use with caution")
    else:
        print("Low trust - Do not deploy without improvements")

trust_metrics(rf, X_test, y_test, None)
